# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JunaidAhamed-7777/flyrank-machine-learning/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane (locked this week):** Refresh Prediction — not reopening the lane.

**Data source:** Same slice as `w03_data_contract.ipynb` — page-level frame with decision date **2026-03-31** on warehouse partitions `month=2026-01` through `month=2026-03` (mid-panel; **not** `fact_content_daily_performance_sample` / `month=2026-06`). If the warehouse is unreachable, the setup cell falls back to `data/raw/content_refresh_anonymized.csv` and states why.

**Rule (plain words):** Rank pages by how stale they are *and* how much search demand they still carry — multiply a staleness weight (full weight at 180+ days since last update, a 0.1 floor otherwise) by trailing `impressions_90d`, tag every row `STALE_HIGH_EXPOSURE` / action `REFRESH`. No trend or label columns enter the score.


## 1. My rule, signal checks, and reason codes

Reason code (fixed): `STALE_HIGH_EXPOSURE`. Action (fixed): `REFRESH`.

Before coding the score, two flag-linked signals get bucket tables and one-word verdicts.


In [1]:
%pip -q install duckdb huggingface_hub truststore pandas numpy

import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

import truststore
truststore.inject_into_ssl()

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "work" / "notebooks"
ROOT = NOTEBOOK_DIR.parent.parent
OUT_DIR = NOTEBOOK_DIR.parent / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

IN_COLAB = "google.colab" in sys.modules
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN and IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    local = NOTEBOOK_DIR.parent / ".hf_token_local"
    if local.is_file():
        HF_TOKEN = local.read_text(encoding="utf-8").strip()
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN missing — use Colab Secret HF_TOKEN, env var, or work/.hf_token_local."
    )

DECISION_DATE = "2026-03-31"
STALE_DAYS = 180
DATA_SOURCE = None
BLOCKED_NOTE = None
con = None

def position_tier(pos: float) -> str:
    if pd.isna(pos) or pos <= 0:
        return "no_data"
    if pos <= 3:
        return "top_3"
    if pos <= 10:
        return "page_1"
    if pos <= 20:
        return "striking"
    if pos <= 50:
        return "page_3_5"
    return "deep"


def load_from_warehouse():
    import duckdb

    global con
    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
    rel = "hf://datasets/FlyRank/internship-warehouse"
    fact_90d = (
        "read_parquet(["
        f"'{rel}/fact_content_daily_performance/month=2026-01/*.parquet',"
        f"'{rel}/fact_content_daily_performance/month=2026-02/*.parquet',"
        f"'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'"
        "])"
    )
    dim_content = f"read_parquet('{rel}/dim_content.parquet')"
    sql = f"""
    WITH daily AS (
        SELECT * FROM {fact_90d}
        WHERE report_date <= DATE '{DECISION_DATE}'
    ),
    agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 90 DAY
                     THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_90d,
            SUM(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 30 DAY
                     THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 60 DAY
                      AND report_date <= DATE '{DECISION_DATE}' - INTERVAL 30 DAY
                     THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_prev30,
            AVG(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 30 DAY
                     THEN gsc_avg_position END) AS avg_position,
            BOOL_AND(gsc_data_available IS TRUE) AS gsc_ok
        FROM daily
        GROUP BY 1, 2
    ),
    labeled AS (
        SELECT
            a.*,
            CASE
                WHEN a.imp_prev30 = 0 AND a.imp_last30 > 0 THEN 'new'
                WHEN a.imp_prev30 = 0 AND a.imp_last30 = 0 THEN 'flat'
                WHEN a.imp_prev30 > 0 AND (a.imp_last30 - a.imp_prev30) * 100.0 / a.imp_prev30 > 20 THEN 'up'
                WHEN a.imp_prev30 > 0 AND (a.imp_last30 - a.imp_prev30) * 100.0 / a.imp_prev30 < -20 THEN 'down'
                ELSE 'stable'
            END AS trend_direction
        FROM agg a
        WHERE a.imp_last30 > 0 AND a.gsc_ok IS TRUE
    )
    SELECT
        l.client_hash_id AS client_id,
        l.content_hash_id AS content_id,
        l.impressions_90d,
        l.avg_position,
        l.trend_direction,
        DATE_DIFF('day', d.content_updated_date, DATE '{DECISION_DATE}') AS days_since_last_update,
        DATE_DIFF('day', d.content_created_date, DATE '{DECISION_DATE}') AS content_age_days,
        d.search_volume
    FROM labeled l
    LEFT JOIN {dim_content} d USING (content_hash_id)
    WHERE DATE_DIFF('day', d.content_created_date, DATE '{DECISION_DATE}') >= 90
    """
    df = con.sql(sql).df()
    return df


def load_from_starter():
    path = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
    df = pd.read_csv(path)
    df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
    return df


try:
    pages = load_from_warehouse()
    DATA_SOURCE = "warehouse_month_2026-03"
    _sample_n = con.sql(
        "SELECT COUNT(*) FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')"
    ).fetchone()[0]
    print(f"Data source: {DATA_SOURCE} (w03 contract slice; decision date {DECISION_DATE}).")
    print(f"Sealed _sample (month=2026-06) rows: {_sample_n:,} — excluded from scoring.")
except Exception as exc:
    BLOCKED_NOTE = str(exc)
    pages = load_from_starter()
    DATA_SOURCE = "starter_csv"
    print(f"BLOCKED: warehouse load failed ({BLOCKED_NOTE[:120]}…)")
    print(f"Falling back to starter_csv: {len(pages):,} rows.")

if "position_tier" not in pages.columns:
    pages["position_tier"] = pages["avg_position"].map(position_tier)

pages["is_declining_label"] = pages["trend_direction"].str.lower().eq("down").astype(int)
print(f"Scoring frame: {len(pages):,} pages | base decline rate: {pages['is_declining_label'].mean():.3f}")


Note: you may need to restart the kernel to use updated packages.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data source: warehouse_month_2026-03 (w03 contract slice; decision date 2026-03-31).
Sealed _sample (month=2026-06) rows: 11,694,072 — excluded from scoring.
Scoring frame: 37,385 pages | base decline rate: 0.210


In [2]:
def decline_by_bucket(frame, bucket_col):
    out = (
        frame.groupby(bucket_col, observed=True)
        .agg(n=("is_declining_label", "size"), decline_rate=("is_declining_label", "mean"))
        .reset_index()
    )
    return out

# --- Signal A: staleness (flag-linked to refresh flags) ---
bins_a = [-np.inf, 30, 90, 180, 365, np.inf]
labels_a = ["<30", "30-90", "90-180", "180-365", "365+"]
pages["staleness_bucket"] = pd.cut(
    pages["days_since_last_update"], bins=bins_a, labels=labels_a
)
signal_a = decline_by_bucket(pages, "staleness_bucket")
print("Signal A — days_since_last_update (staleness)")
display(signal_a)

# --- Signal B: search_volume (flag-linked to quick-win) ---
sv = pages["search_volume"].fillna(0)
pages["volume_bucket"] = "zero"
pos = sv > 0
if pos.any():
    edges = sv[pos].quantile([0, 1 / 3, 2 / 3, 1]).to_numpy()
    edges[0], edges[-1] = -1, np.inf
    pages.loc[pos, "volume_bucket"] = pd.cut(
        sv[pos], bins=edges, labels=["low", "mid", "high"], include_lowest=True
    ).astype(str)
signal_b = decline_by_bucket(pages, "volume_bucket")
print("Signal B — search_volume")
display(signal_b)

SIGNAL_A_VERDICT = "MIXED"
SIGNAL_B_VERDICT = "FALSE"


Signal A — days_since_last_update (staleness)


,staleness_bucket,n,decline_rate
0,<30,29866,0.188006
1,30-90,7464,0.298499
2,90-180,51,0.000000
3,180-365,4,0.000000


Signal B — search_volume


,volume_bucket,n,decline_rate
0,high,7877,0.213787
1,low,10916,0.204012
2,mid,5651,0.195187
3,zero,12941,0.218608


**Signal A (`days_since_last_update`, refresh-flag input):** **MIXED**

In the bulk of pages, decline rate rises from the `<30` bucket into `30-90` (higher staleness → higher measured decline). Buckets at `90+` days are tiny on this March slice and do not sustain a clean monotonic story at the **180-day** threshold the rule uses; many high-traffic rows also show negative days-since-update (update timestamp after the decision anchor), which weakens the flag linkage. The rule still uses the 180-day threshold as assigned, but the audit is not a clean CONFIRMED.

**Signal B (`search_volume`, quick-win flag input):** **FALSE**

Decline rates are flat across zero/low/mid/high volume buckets (all near the ~21% base rate). That is a win: **this baseline will not lean on search volume.**


## 2. Build the ranked queue (writes the CSV)

Score: `staleness_flag * impressions_90d` with `staleness_flag = 1` if `days_since_last_update >= 180`, else `0.1`. Reason: `STALE_HIGH_EXPOSURE`. Action: `REFRESH`.


In [3]:
REASON_CODE = "STALE_HIGH_EXPOSURE"
ACTION_LABEL = "REFRESH"
RULE_NAME = "staleness_flag_x_impressions_90d"

pages["staleness_flag"] = np.where(pages["days_since_last_update"] >= STALE_DAYS, 1.0, 0.1)
pages["baseline_score"] = pages["staleness_flag"] * pages["impressions_90d"]
pages["reason_code"] = REASON_CODE
pages["action"] = ACTION_LABEL

queue = pages.sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

top50_rate = float(queue.head(50)["is_declining_label"].mean())
base_rate = float(queue["is_declining_label"].mean())

csv_path = OUT_DIR / "baseline_action_score.csv"
json_path = OUT_DIR / "baseline_metrics.json"

export_cols = [
    "rank",
    "content_id",
    "client_id",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "position_tier",
    "trend_direction",
    "is_declining_label",
]
queue[export_cols].to_csv(csv_path, index=False)

metrics = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "data_source": DATA_SOURCE,
    "blocked_note": BLOCKED_NOTE,
    "row_count_scored": int(len(queue)),
    "rule_name": RULE_NAME,
    "score_formula": "staleness_flag * impressions_90d; staleness_flag=1 if days_since_last_update>=180 else 0.1",
    "staleness_threshold_days": STALE_DAYS,
    "reason_code": REASON_CODE,
    "action_label": ACTION_LABEL,
    "base_decline_rate": base_rate,
    "precision_at_50_top50_decline_rate": top50_rate,
    "signal_a_verdict": SIGNAL_A_VERDICT,
    "signal_b_verdict": SIGNAL_B_VERDICT,
    "caveat": "Score ignores trend_direction; top ranks are often high-impression pages with soft 0.1 staleness weight, not true 180d+ stale URLs.",
}
json_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(f"Wrote {csv_path} ({len(queue):,} rows)")
print(f"Wrote {json_path}")
print(f"Base decline rate: {base_rate:.3f} | Precision@50 (top-50 decline rate): {top50_rate:.3f}")

top10 = queue.head(10)
display(top10[["rank", "content_id", "baseline_score", "days_since_last_update", "impressions_90d", "position_tier", "trend_direction"]])


Wrote C:\aaFiles\aaGithub\FlyrankAI\flyrank-machine-learning\work\outputs\baseline_action_score.csv (37,385 rows)
Wrote C:\aaFiles\aaGithub\FlyrankAI\flyrank-machine-learning\work\outputs\baseline_metrics.json
Base decline rate: 0.210 | Precision@50 (top-50 decline rate): 0.240


,rank,content_id,baseline_score,days_since_last_update,impressions_90d,position_tier,trend_direction
0,1,content_e241d6415ac9e534,50968.4,-84,509684.0,page_1,stable
1,2,content_e8a52cf3d5988c07,50813.9,-72,508139.0,striking,up
2,3,content_e7b5dd4dff461ad2,50720.3,-72,507203.0,page_1,stable
3,4,content_512dbad65bd5ade9,49274.8,-87,492748.0,page_1,stable
4,5,content_b99ea6861864dea5,46321.8,-94,463218.0,page_1,stable
5,6,content_acbcc847f8996314,43309.5,-94,433095.0,page_1,stable
6,7,content_00d4fdf6e48a2d38,40760.4,-97,407604.0,page_1,stable
7,8,content_471d9cabce329a66,39342.1,-87,393421.0,page_1,up
8,9,content_4d0d79fc12632ef8,38873.7,-87,388737.0,page_1,down
9,10,content_7172a7fad43f0998,36920.4,-94,369204.0,page_1,up
